In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import gc
import os
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import MatchEnv, PoolController, RandomController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomReset
from src.rl.reward_shapers import DenseReward_3
from src.rl.trainer import train_ppo, export_huggingface

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

⚡ Device: cuda


In [5]:
# ── CONFIGURATION ──
STAGE = 3
SAVE_DIR = f"models/stage{STAGE}"
POOL_DIR = f"models/stage{STAGE}/pool"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(POOL_DIR, exist_ok=True)

MAX_STEPS = 1800  # 30.0s
TIME_LIMIT = 30.0
NUM_ENVS = 16


def make_env(env_idx: int):
    def _init():
        import torch
        torch.set_num_threads(1) 
        
        is_red = env_idx % 2 == 0
        learner_team = "red" if is_red else "blue"
        opp_team = "blue" if is_red else "red"

        opp_ctrl = PoolController(pool_dir=POOL_DIR, device="cpu")

        if is_red:
            roster = [
                PlayerSlot("red", PlayerStats(name="Learner", accel=3200.0), controller="RL"),
                PlayerSlot("blue", PlayerStats(name="Opponent", accel=3200.0), controller=opp_ctrl),
            ]
        else:
            roster = [
                PlayerSlot("red", PlayerStats(name="Opponent", accel=3200.0), controller=opp_ctrl),
                PlayerSlot("blue", PlayerStats(name="Learner", accel=3200.0), controller="RL"),
            ]

        cfg = MatchConfig(mode=ClassicMatchMode(time_limit=TIME_LIMIT, score_limit=99), roster=roster)

        return MatchEnv(
            match_config=cfg,
            reward_shaper=DenseReward_3(team=learner_team),
            reset_strategy=RandomReset(),
            learner_team=learner_team,
            max_steps=MAX_STEPS,
        )
    return _init

# CRITICAL: Use AsyncVectorEnv with "spawn" to engage all CPU cores safely
train_envs = gym.vector.AsyncVectorEnv(
    [make_env(i) for i in range(NUM_ENVS)],
    context="spawn" 
)

model = ActorCritic(obs_dim=80).to(device)

stage2_best = "models/stage2/best_model.pt"
if os.path.exists(stage2_best):
    model.load_state_dict(torch.load(stage2_best, map_location=device, weights_only=False))
    print(f"✅ Loaded checkpoint from Stage 2")

✅ Loaded checkpoint from Stage 2


In [6]:
train_ppo(
    envs=train_envs,
    model=model,
    device=device,
    max_steps=MAX_STEPS,
    time_limit=TIME_LIMIT,
    baseline_type="heuristic",
    double_eval=True,              
    previous_model_path=stage2_best, 
    total_timesteps=1_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
    lr_initial=3e-5,
    lr_final=5e-6,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef_initial=0.006,
    ent_coef_final=0.002,
)

train_envs.close()

🚀 Training: Target [HEURISTIC] | Batch: 4096 | Eval: 50 eps

📊 [EVALUATION @ Step  102400 | Target: HEURISTIC]
   Scoring Episodes: 43/50 (86.0%)
   Goals [Scored: 104 | Conceded: 12 | Net: +92]
   Avg Speed: 9.48s | Win Rate: 84.0% | Avg Net: +1.84
   ❌ FAILED SANITY CHECK [Win Rate: 84.0% < 85.0%]. Skipping champion trial.

📊 [EVALUATION @ Step  200704 | Target: HEURISTIC]
   Scoring Episodes: 48/50 (96.0%)
   Goals [Scored: 114 | Conceded: 10 | Net: +104]
   Avg Speed: 10.48s | Win Rate: 94.0% | Avg Net: +2.08
   ⚔️ [DOUBLE EVAL] Passed sanity check! Testing against Champion...
   ⚔️ Goals [Scored: 31 | Conceded: 35 | Net: -4] | Win Rate: 18.0%
   ❌ RETAINING CHAMPION. Margin not decisive enough (Net: -4 < +6).

📊 [EVALUATION @ Step  303104 | Target: HEURISTIC]
   Scoring Episodes: 44/50 (88.0%)
   Goals [Scored: 100 | Conceded: 14 | Net: +86]
   Avg Speed: 11.70s | Win Rate: 80.0% | Avg Net: +1.72
   ❌ FAILED SANITY CHECK [Win Rate: 80.0% < 85.0%]. Skipping champion trial.

📊 [EVAL

KeyboardInterrupt: 

In [ ]:
if os.path.exists(f"{SAVE_DIR}/best_model.pt"):
    export_huggingface(
        model_path=f"{SAVE_DIR}/best_model.pt",
        save_dir=SAVE_DIR,
        device=device,
        stage=STAGE,
        time_limit=TIME_LIMIT,
        max_steps=MAX_STEPS,
        eval_episodes=100
    )

In [20]:
import os
import sys
import torch

sys.path.insert(0, os.path.abspath(".."))
from src.rl.evaluator import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Render 5 evaluation matches of your best checkpoint
replay_file = evaluate_and_generate_html(
    model_or_path="models/stage3/best_model.pt",
    device=device,
    baseline_type="heuristic",  # Visualizes matches against Heuristic bot
    output_dir="render/",
    filename="stage3_diagnostic.html",
    num_episodes=10,
    max_steps=1800,  # 30 seconds per match
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training/render/stage3_diagnostic.html


In [7]:
import os
import sys
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import PlayerSlot, PlayerStats
from src.rl.benchmarker import RLController, run_arena
from src.rl.ppo_core import ActorCritic

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load Both Checkpoints
path_model_a = "models/stage2/best_model.pt"
path_model_b = "models/stage3/best_model.pt"

name1 = "stage2_model"
name2 = "stage3_model"

model_a = ActorCritic(obs_dim=80).to(device)
model_a.load_state_dict(
    torch.load(path_model_a, map_location=device, weights_only=False)
)
model_a.eval()

model_b = ActorCritic(obs_dim=80).to(device)
model_b.load_state_dict(
    torch.load(path_model_b, map_location=device, weights_only=False)
)
model_b.eval()

# 2. Round 1: Model A (Red) vs Model B (Blue)
ctrl_a_red = RLController(
    model_a, team="red", device=device, deterministic=True
)
ctrl_b_blue = RLController(
    model_b, team="blue", device=device, deterministic=True
)

roster_red_r1 = [
    PlayerSlot(
        "red", PlayerStats(name=name1, accel=3200.0), ctrl_a_red
    )
]
roster_blue_r1 = [
    PlayerSlot(
        "blue", PlayerStats(name=name2, accel=3200.0), ctrl_b_blue
    )
]

print(f"⚔️ [ROUND 1] {name1} (RED) vs {name2} (BLUE)")
stats_r1 = run_arena(
    roster_red_r1, roster_blue_r1, num_matches=5, time_limit=60.0, score_limit=3
)

# 3. Round 2: Side Swap to Eliminate Field/Spawn Bias
ctrl_b_red = RLController(
    model_b, team="red", device=device, deterministic=True
)
ctrl_a_blue = RLController(
    model_a, team="blue", device=device, deterministic=True
)

roster_red_r2 = [
    PlayerSlot(
        "red", PlayerStats(name=name2, accel=3200.0), ctrl_b_red
    )
]
roster_blue_r2 = [
    PlayerSlot(
        "blue", PlayerStats(name=name1, accel=3200.0), ctrl_a_blue
    )
]

print(f"\n⚔️ [ROUND 2 - SWAPPED] {name2} (RED) vs {name1} (BLUE)")
stats_r2 = run_arena(
    roster_red_r2, roster_blue_r2, num_matches=5, time_limit=60.0, score_limit=3
)

# 4. Aggregated Match Statistics
wins_a = stats_r1["RED_WINS"] + stats_r2["BLUE_WINS"]
wins_b = stats_r1["BLUE_WINS"] + stats_r2["RED_WINS"]
draws = stats_r1["DRAWS"] + stats_r2["DRAWS"]
goals_a = stats_r1["RED_GOALS"] + stats_r2["BLUE_GOALS"]
goals_b = stats_r1["BLUE_GOALS"] + stats_r2["RED_GOALS"]

print("\n" + "=" * 55)
print("📊 FINAL HEAD-TO-HEAD RESULT (10 Matches Total)")
print(
    f"  • {name1}   : {wins_a:2d} Wins | {goals_a:3d} Goals Scored (Net:"
    f" {goals_a - goals_b:+d})"
)
print(
    f"  • {name2} : {wins_b:2d} Wins | {goals_b:3d} Goals Scored (Net:"
    f" {goals_b - goals_a:+d})"
)
print(f"  • Draws        : {draws:2d}")
print("=" * 55)

⚔️ [ROUND 1] stage2_model (RED) vs stage3_model (BLUE)
🏟️ Running Arena: 1 RED vs 1 BLUE (5 Matches)
✅ Completed in 81.96s
🏆 Wins: RED 5 | BLUE 0 | DRAWS 0

⚔️ [ROUND 2 - SWAPPED] stage3_model (RED) vs stage2_model (BLUE)
🏟️ Running Arena: 1 RED vs 1 BLUE (5 Matches)
✅ Completed in 86.75s
🏆 Wins: RED 5 | BLUE 0 | DRAWS 0

📊 FINAL HEAD-TO-HEAD RESULT (10 Matches Total)
  • stage2_model   :  5 Wins |   5 Goals Scored (Net: +0)
  • stage3_model :  5 Wins |   5 Goals Scored (Net: +0)
  • Draws        :  0
